In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Backpressure CHP

This example demonstrates how to model a Combined Heat and Power (CHP) plant with a fixed heat-power ratio, assuming that the plant is operated in backpressure mode.
For an example of a CHP plant with a more complicated heat-power feasible operational area, see the [extraction-condensing CHP example](https://docs.pypsa.org/latest/examples/power-to-gas-boiler-chp/). In this example, the CHP is modelled as a `Link` component with two output buses: one for electricity and one for heat. The CHP unit must be heat-following since there is no other supply of heat to the "Frankfurt heat" bus.

In [ ]:
import pypsa

n = pypsa.Network()

n.add("Bus", "Frankfurt", carrier="AC")
n.add("Load", "Frankfurt", bus="Frankfurt", p_set=5)

n.add("Bus", "Frankfurt heat", carrier="heat")
n.add("Load", "Frankfurt heat", bus="Frankfurt heat", p_set=3)

n.add("Bus", "Frankfurt gas", carrier="gas")
n.add("Generator", "Frankfurt gas", bus="Frankfurt gas", marginal_cost=100, p_nom=100)

n.add(
    "Link",
    "OCGT",
    bus0="Frankfurt gas",
    bus1="Frankfurt",
    p_nom_extendable=True,
    capital_cost=600,
    efficiency=0.4,  # electricity per unit of gas
)

n.add(
    "Link",
    "CHP",
    bus0="Frankfurt gas",
    bus1="Frankfurt",
    bus2="Frankfurt heat",
    p_nom_extendable=True,
    capital_cost=1400,
    efficiency=0.3,  # electricity per unit of gas
    efficiency2=0.3,  # heat per unit of gas
)

n.optimize();

In [ ]:
n.loads_t.p

In [ ]:
n.links_t.p0

In [ ]:
n.links_t.p1

In [ ]:
n.links_t.p2

## Levelised cost of each output

The CHP sells electricity and heat, so its levelised cost depends on the output it is divided by. Levelised over electricity, its heat sales are credited at the heat price, and vice versa. Without `bus_carrier`, its costs are spread over both outputs (€/MWh):

In [ ]:
import pandas as pd

pd.DataFrame(
    {
        "electricity": n.statistics.lcoe(bus_carrier="AC", groupby=False),
        "heat": n.statistics.lcoe(bus_carrier="heat", groupby=False),
        "both": n.statistics.lcoe(groupby=False),
    }
).loc["Link"]

In equilibrium, the levelised cost of each output equals its market value, here the price of that output ([Brown and Reichenberg, 2021](https://doi.org/10.1016/j.eneco.2021.105354)):

In [ ]:
n.buses_t.marginal_price